# Day 3: phase patterns, undoing QFT, and phase estimation

Material development: **Gage Plott**.

This complete worked example has two distinct parts. First, we prepare a known
basis state, apply the quantum Fourier transform (QFT), inspect its simulated
complex amplitudes, and undo it. Then we use quantum phase estimation (QPE) to
learn an eigenphase of a supplied operation. Undoing a QFT by itself is **not**
QPE: QPE also needs a prepared target eigenstate and controlled powers of the
operation.

The rightmost displayed bit is qubit 0. For example, `01` sets qubit 0 and `10`
sets qubit 1. Run from top to bottom in the Python (PHYS 3810 Guided) kernel. Only the
two marked parameter cells invite edits. The other code is a worked display
and checking pipeline; learners do not need to write arrays, loops, or matrices.

## 1. Run support imports unchanged

The path code locates this course's own library inside the repository or the
extracted `runtime` folder. Keep the folder together.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
from importlib.metadata import version

# Locate only this extracted runtime/repository tree, never a fixed developer path.
repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "phys3810" / "grover.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Keep the runtime folder together and launch Jupyter from its root.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12)
assert version("qiskit") == "2.6.1", "Select the pinned classroom kernel."

from cmath import exp  # noqa: E402 - after standalone path/environment setup
from math import isclose, pi  # noqa: E402 - after standalone path/environment setup

import matplotlib.pyplot as plt  # noqa: E402 - after standalone path/environment setup
from IPython.display import display  # noqa: E402 - after standalone path/environment setup
from qiskit import ClassicalRegister, QuantumCircuit  # noqa: E402 - after standalone path/environment setup
from qiskit.circuit.library import PhaseGate  # noqa: E402 - after standalone path/environment setup
from qiskit.primitives import StatevectorSampler  # noqa: E402 - after standalone path/environment setup
from qiskit.quantum_info import Statevector  # noqa: E402 - after standalone path/environment setup
from qiskit.visualization import plot_histogram  # noqa: E402 - after standalone path/environment setup
from src.phys3810.qft import qft_circuit  # noqa: E402 - after standalone path/environment setup
from src.phys3810.qpe import qpe_circuit  # noqa: E402 - after standalone path/environment setup

print("Course source:", Path(sys.modules[qpe_circuit.__module__].__file__).resolve())


## 2. QFT: select a known input label

**Editable:** use `basis_label = "01"` first, then `"10"`. Keep the two-qubit
width fixed. After editing, run the parameter cell itself with **Shift+Enter**,
then run the following code cells in order. Editing text alone does not update
Python's variables.

For this course's forward QFT, input index $k$ becomes amplitudes
$e^{+2\pi i j k/4}/2$ at output indices $j=0,1,2,3$. The **positive** sign is
our Fourier convention. The inverse uses the negative sign, and both include
the swaps needed for ordinary displayed binary order. Every output probability
is $1/4$, yet the phases depend on $k$. A histogram of these probabilities alone
cannot reveal the phase pattern.


In [ ]:
basis_label = "01"
assert basis_label in ("01", "10"), "Use 01 or 10 in this guided example."


The first three lines prepare the named basis state, build the canonical QFT,
and apply it in the simulator. The remaining lines are read-only mathematical checks using
the formula above. Here Python writes the imaginary unit as `1j`; the output
index `j` in the equation is a separate mathematical symbol.


In [ ]:
qft_input = Statevector.from_label(basis_label)
forward = qft_circuit(2, inverse=False, do_swaps=True)
fourier_state = qft_input.evolve(forward)
amplitudes = fourier_state.data
basis_index = int(basis_label, 2)
expected_amplitudes = [
    1 / 2,
    exp(2j * pi * basis_index / 4) / 2,
    exp(2j * pi * 2 * basis_index / 4) / 2,
    exp(2j * pi * 3 * basis_index / 4) / 2,
]
# Read-only check: compare all four complex amplitudes with the independent formula.
assert all(abs(actual - expected) < 1e-12
           for actual, expected in zip(amplitudes, expected_amplitudes))
assert all(isclose(float(p), 1 / 4, rel_tol=0, abs_tol=1e-12)
           for p in fourier_state.probabilities())
print(f"QFT input |{basis_label}>; output probabilities:", fourier_state.probabilities())
print("Complex amplitudes in output order 00, 01, 10, 11:", amplitudes)


## 3. Inspect phase-sensitive information

A complex amplitude has real and imaginary components. This plot shows both;
negative values and imaginary components are meaningful, not negative
probabilities. For input `01`, the amplitudes are $1/2, i/2, -1/2, -i/2$.
For `10`, they are $1/2, -1/2, 1/2, -1/2$. Both give the same four probabilities.

This simulator display reads amplitudes directly. A single computational-basis
measurement on hardware would not reveal all these components. Relative phase
has to be probed through suitable interference/measurement procedures.


In [ ]:
phase_title = f"QFT of |{basis_label}>: real and imaginary amplitudes"
figure, phase_axes = plt.subplots(figsize=(8, 3.8))
phase_axes.bar([-0.16, 0.84, 1.84, 2.84], amplitudes.real, width=0.32, label="Real component")
phase_axes.bar([0.16, 1.16, 2.16, 3.16], amplitudes.imag, width=0.32, label="Imaginary component")
phase_axes.axhline(0, color="black", linewidth=0.8)
phase_axes.set_xticks([0, 1, 2, 3], ["00", "01", "10", "11"])
phase_axes.set(xlabel="Output basis label", ylabel="Amplitude component", ylim=(-0.65, 0.75),
               title=phase_title)
phase_axes.legend()
figure.tight_layout()
display(figure)
plt.close(figure)


## 4. Undo this QFT

**Predict:** does undoing the QFT recover one label or four equally likely
labels? The inverse combines the phase information through interference and
recovers the original basis state. We apply it to the unmeasured Fourier state,
not to a state collapsed by an intervening measurement.

These are ordinary basis indices. Do **not** divide the recovered QFT label by
$2^m$ and call it a measured phase. Phase decoding belongs to the next part's
QPE counting register.


In [ ]:
recovered = fourier_state.evolve(qft_circuit(2, inverse=True, do_swaps=True))
recovery_probabilities = recovered.probabilities()
assert isclose(float(recovery_probabilities[basis_index]), 1.0, rel_tol=0, abs_tol=1e-12)
recovery_title = f"Inverse QFT recovers |{basis_label}>"
figure, recovery_axes = plt.subplots(figsize=(8, 3.5))
recovery_axes.bar(["00", "01", "10", "11"], recovery_probabilities)
recovery_axes.set(xlabel="Recovered basis label", ylabel="Exact probability", ylim=(0, 1.15),
                  title=recovery_title)
figure.tight_layout()
display(figure)
plt.close(figure)
print(f"Recovered input |{basis_label}> with probability", float(recovery_probabilities[basis_index]))


## 5. QPE: choose an exact eigenphase

**Editable:** `exact_phase` may be `3 / 8` or `1 / 8`. Keep three counting
qubits, one target qubit, 512 samples, and seed 2026. After editing, run this
parameter cell itself with **Shift+Enter**, then run the following QPE code
cells in order. Editing text alone does not update Python's variables.

The one-qubit operation is a phase gate $U=P(2\pi\phi)$:
$U|1\rangle=e^{2\pi i\phi}|1\rangle$. We explicitly prepare the **target in
$|1\rangle$**. Preparing it in $|0\rangle$ instead would give eigenphase zero.

Counting qubits 0, 1, 2 start at zero; target qubit 3 is separate. QPE puts the
counting qubits in superposition, applies controlled powers $U^1,U^2,U^4$, and
uses inverse QFT to decode the phase. The counting register has eight labels,
so its exact grid is in steps of $1/8$.

**Predict:** $3/8$ gives `011`, while $1/8$ gives `001`. Only for this QPE
counting register do we decode `int(bits, 2) / 8`. Count qubit 0 is the
rightmost bit; no extra reversal is applied. These asymmetric labels help us
notice reversed bit order.


In [ ]:
exact_phase = 3 / 8
count_width = 3
shots = 512
seed = 2026
assert exact_phase in (1 / 8, 3 / 8), "Use 1/8 or 3/8 in this exact-phase route."


Run this worked construction unchanged. `qpe_circuit` supplies the tested
counting/target registers and controlled operations, but it does not prepare
the target eigenstate for us. The explicit X below prepares target $|1\rangle$.
A new circuit is created each time this cell runs.


In [ ]:
unitary = PhaseGate(2 * pi * exact_phase)
qpe_block = qpe_circuit(count_width, unitary)
qpe_prepared = QuantumCircuit(*qpe_block.qregs)
qpe_prepared.x(qpe_block.qregs[1][0])  # Explicitly prepare the target eigenstate |1>.
qpe_prepared.compose(qpe_block, inplace=True)
expected_index = int(exact_phase * 2**count_width)
expected_label = format(expected_index, f"0{count_width}b")
counting_probabilities = Statevector.from_instruction(qpe_prepared).probabilities([0, 1, 2])
assert isclose(float(counting_probabilities[expected_index]), 1.0, rel_tol=0, abs_tol=2e-11)
assert qpe_prepared.num_clbits == 0
print("Registers: count[0], count[1], count[2], then target[0].")
print(f"Phase {exact_phase:g}; expected counting label {expected_label}.")


In [ ]:
qpe_circuit_title = f"Three count qubits; target |1>; phase={exact_phase:g}"
figure = qpe_prepared.draw("mpl", fold=24)
figure.suptitle(qpe_circuit_title)
display(figure)
plt.close(figure)


## 6. Measure only the counting register

We add three classical bits named `phase` and measure count qubits 0, 1, 2
into matching classical positions. The target is **not** measured into this
result. These exact examples are deterministic in the ideal simulator; finite
sampling does not broaden a distribution that has probability one at one label.


In [ ]:
qpe_measured = qpe_prepared.copy()
phase_register = ClassicalRegister(count_width, "phase")
qpe_measured.add_register(phase_register)
qpe_measured.measure([0, 1, 2], phase_register)
phase_counts = StatevectorSampler(seed=seed).run(
    [qpe_measured], shots=shots
).result()[0].data.phase.get_counts()
assert phase_counts == {expected_label: shots}
assert qpe_prepared.num_clbits == 0
observed_label = next(iter(phase_counts))
decoded_phase = int(observed_label, 2) / 2**count_width
assert isclose(decoded_phase, exact_phase, rel_tol=0, abs_tol=1e-12)
phase_histogram_title = f"QPE: {expected_label} decodes to {decoded_phase:g}; {shots} samples"
figure = plot_histogram(phase_counts, figsize=(8, 3.5), title=phase_histogram_title)
display(figure)
plt.close(figure)
print("Measured counting register:", phase_counts)
print(f"{observed_label} -> {int(observed_label, 2)}/{2**count_width} = {decoded_phase:g}")


## 7. Change, rerun, and discuss

First change the QFT input from `"01"` to `"10"`. Run that parameter cell itself
with **Shift+Enter**, then the forward, pattern, and inverse cells in order.
The complex pattern changes although all four Fourier probabilities remain
one quarter; inverse QFT still recovers the selected input.
Then change `exact_phase` from `3 / 8` to `1 / 8`. Execute that parameter cell
itself, then the QPE build, circuit, and sampling cells in order. The result
changes from `011` to `001` and decodes to the selected phase. Editing text
alone does not update Python's variables.

On the **50-minute route**, finish the QFT pattern and inverse recovery, then
stop; defer QPE. Only if additional time is available, the presenter may show
the default QPE example as an optional demonstration outside that route. Keep
nonexact phases, mixed target states, SDK convention comparisons, and advanced
derivations outside this short route. In a two-day schedule, omit this session and retain Day 1 plus Grover;
do not squeeze every topic into fewer minutes.

HHL and the compiled N=15 Shor example remain available as reference lessons.
Their restrictions are unchanged. Shor would replace an agreed segment rather
than being appended to an already full session. QPE here uses one target qubit;
the general course helper supports only one or two, and Shor has its own separate
four-residue-qubit builder. Small local simulations do not demonstrate quantum
advantage or remove the costs of state preparation, controlled operations,
and readout. Save your working copy; restart and run all if state becomes stale.
